# 01. document/ -> ChromaDB 임베딩

이 노트북은 최초 1회 또는 `document/` 문서가 변경됐을 때 실행한다.

- 입력: `document/` 내부 HTML/PDF/TXT/MD
- 출력: `agent_data/chroma` ChromaDB 컬렉션
- 런타임 RAG: `manufacturing_agent.ipynb`의 Evidence RAG가 이 컬렉션을 읽는다.


In [8]:
from __future__ import annotations

import os
import re
import hashlib
from pathlib import Path

from bs4 import BeautifulSoup
import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
from chromadb.utils import embedding_functions

DATA_DIR = "agent_data"
DOCUMENT_DIR = "document"
CHROMA_DIR = os.path.join(DATA_DIR, "chroma")
EMBED_MODEL = "text-embedding-3-small"

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 180
LOCAL_EMBED_DIM = 384

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)


def load_dotenv(path: str = ".env") -> None:
    env_path = Path(path)
    if not env_path.exists():
        return
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_dotenv()
EMBED_MODEL = os.environ.get("OPENAI_EMBED_MODEL", EMBED_MODEL)
_HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print("ChromaDB path:", CHROMA_DIR)
print("Document path:", DOCUMENT_DIR)
print("OpenAI embedding key:", _HAS_KEY)


ChromaDB path: agent_data\chroma
Document path: document
OpenAI embedding key: True


In [9]:
class LocalHashEmbeddingFunction(EmbeddingFunction[Documents]):
    """외부 모델 다운로드 없이 동작하는 로컬 임베딩 함수."""

    def __call__(self, input: Documents) -> Embeddings:
        vectors = []
        for text in input:
            vec = [0.0] * LOCAL_EMBED_DIM
            tokens = re.findall(r"[A-Za-z가-힣0-9_]+", text.lower())
            for token in tokens:
                digest = hashlib.sha256(token.encode("utf-8")).digest()
                idx = int.from_bytes(digest[:4], "little") % LOCAL_EMBED_DIM
                sign = 1.0 if digest[4] % 2 == 0 else -1.0
                vec[idx] += sign
            norm = sum(v * v for v in vec) ** 0.5 or 1.0
            vectors.append([v / norm for v in vec])
        return vectors


def build_embedding_function():
    """manufacturing_agent.ipynb의 런타임 임베딩 함수와 동일해야 한다."""
    if _HAS_KEY:
        return embedding_functions.OpenAIEmbeddingFunction(
            api_key=os.environ["OPENAI_API_KEY"], model_name=EMBED_MODEL
        ), "manufacturing_document_chunks_openai", f"OpenAI({EMBED_MODEL})"
    return LocalHashEmbeddingFunction(), "manufacturing_document_chunks_local_hash", f"LocalHash({LOCAL_EMBED_DIM})"


embed_fn, collection_name, embed_label = build_embedding_function()
print("Collection:", collection_name)
print("Embedding:", embed_label)


Collection: manufacturing_document_chunks_openai
Embedding: OpenAI(text-embedding-3-small)


In [10]:
def doc_type(path: Path) -> str:
    parts = {p.lower() for p in path.parts}
    name = path.name.lower()
    if "osha" in parts or "kosha" in parts or "safety" in name or "loto" in name or "guard" in name:
        return "safety"
    if "haas" in parts or "troubleshooting" in name or "diagnostic" in name:
        return "troubleshooting"
    return "concept"


def read_html(path: Path) -> str:
    raw = path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "footer", "aside"]):
        tag.decompose()
    return soup.get_text("\n")


def read_pdf(path: Path) -> str:
    try:
        from pypdf import PdfReader
    except ImportError as e:
        raise RuntimeError("PDF 임베딩에는 pypdf가 필요합니다. `uv pip install pypdf` 후 다시 실행하세요.") from e
    reader = PdfReader(str(path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def clean_text(text: str) -> str:
    lines = [re.sub(r"\s+", " ", line).strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if len(line) >= 2)


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    text = clean_text(text)
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks


In [11]:
def load_document_chunks(document_dir: str = DOCUMENT_DIR) -> list[dict]:
    root = Path(document_dir)
    supported = sorted(
        p for p in root.rglob("*")
        if p.suffix.lower() in {".html", ".htm", ".pdf", ".txt", ".md"}
    )
    chunks: list[dict] = []
    for path in supported:
        suffix = path.suffix.lower()
        if suffix in {".html", ".htm"}:
            text = read_html(path)
        elif suffix == ".pdf":
            text = read_pdf(path)
        else:
            text = path.read_text(encoding="utf-8", errors="ignore")

        rel = path.relative_to(root).as_posix()
        for idx, chunk in enumerate(chunk_text(text)):
            digest = hashlib.sha1(f"{rel}:{idx}:{chunk[:80]}".encode("utf-8")).hexdigest()[:16]
            chunks.append({
                "id": digest,
                "text": chunk,
                "metadata": {
                    "source": rel,
                    "chunk_index": idx,
                    "type": doc_type(path),
                    "ext": suffix.lstrip("."),
                },
            })
    return chunks


chunks = load_document_chunks()
print("Loaded chunks:", len(chunks))
print("First chunk source:", chunks[0]["metadata"]["source"] if chunks else "none")


Loaded chunks: 213
First chunk source: haas/Mill Chatter - Troubleshooting - TG0100.html


In [15]:
def get_collection(reset: bool = False):
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    if reset:
        try:
            client.delete_collection(collection_name)
        except Exception:
            pass
    return client.get_or_create_collection(collection_name, embedding_function=embed_fn)


def check_embedding_status(reset: bool = False) -> dict:
    collection = get_collection(reset=reset)
    existing_ids = set(collection.get(include=[])["ids"]) if collection.count() else set()
    target_ids = {chunk["id"] for chunk in chunks}
    new_chunks = [chunk for chunk in chunks if chunk["id"] not in existing_ids]
    stale_ids = existing_ids - target_ids
    status = {
        "collection": collection_name,
        "embedding": embed_label,
        "stored_chunks": collection.count(),   #저장된 청크 수
        "document_chunks": len(chunks),        #문서에서 읽은 청크 수
        "new_chunks": len(new_chunks),         #새로 임베딩이 필요한 청크 수
        "stale_chunks": len(stale_ids),        #ChromaDB에 남아 있는 청크 수
        "is_fully_embedded": len(new_chunks) == 0 and len(chunks) > 0,
    }
    print("임베딩 상태 확인")
    for key, value in status.items():
        print(f"{key}: {value}")
    if stale_ids:
        print("주의: document/에서 사라진 chunk가 ChromaDB에 남아 있습니다. 정리가 필요하면 다음 셀에서 reset=True로 실행하세요.")
    return {"collection": collection, "new_chunks": new_chunks, "status": status}


# 임베딩 전에 먼저 현재 상태를 확인한다. 전체 재임베딩이 필요하면 reset=True로 바꾼다.
embedding_plan = check_embedding_status(reset=False)


임베딩 상태 확인
collection: manufacturing_document_chunks_openai
embedding: OpenAI(text-embedding-3-small)
stored_chunks: 213
document_chunks: 213
new_chunks: 0
stale_chunks: 0
is_fully_embedded: True


In [ ]:
def embed_documents_to_chroma(plan: dict):
    collection = plan["collection"]
    new_chunks = plan["new_chunks"]

    batch_size = 64
    for i in range(0, len(new_chunks), batch_size):
        batch = new_chunks[i:i + batch_size]
        collection.add(
            ids=[chunk["id"] for chunk in batch],
            documents=[chunk["text"] for chunk in batch],
            metadatas=[chunk["metadata"] for chunk in batch],
        )

    print("ChromaDB 임베딩 완료")
    print("collection:", collection_name)
    print("embedding:", embed_label)
    print("total:", collection.count())
    print("added:", len(new_chunks))
    return collection


# 아래 점검 셀(#2~#7: 샘플 확인 / source·type 집계 / query / vector 차원)에서
# 사용할 수 있도록 collection을 전역 변수로 노출한다.
collection = embed_documents_to_chroma(embedding_plan)

In [18]:
# 2. 저장된 문서 chunk 샘플 확인
from collections import Counter

sample = collection.get(
    limit=5,
    include=["documents", "metadatas"]
)

for i, chunk_id in enumerate(sample["ids"]):
    print("=" * 80)
    print("id:", chunk_id)
    print("metadata:", sample["metadatas"][i])
    print("document preview:")
    print(sample["documents"][i][:700])

NameError: name 'collection' is not defined

In [19]:
# 3. source 문서별 chunk 개수 확인


all_data = collection.get(include=["metadatas"])
source_counts = Counter(meta["source"] for meta in all_data["metadatas"])

for source, count in source_counts.most_common():
    print(f"{source}: {count} chunks")

NameError: name 'collection' is not defined

In [20]:
# 4. 문서 type별 chunk 개수 확인

type_counts = Counter(meta["type"] for meta in all_data["metadatas"])

for doc_type_name, count in type_counts.items():
    print(f"{doc_type_name}: {count} chunks")

NameError: name 'all_data' is not defined

In [13]:
# 5. 실제 검색이 되는지 확인

query = "spindle vibration troubleshooting"

results = collection.query(
    query_texts=[query],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

for i in range(len(results["ids"][0])):
    print("=" * 80)
    print("rank:", i + 1)
    print("id:", results["ids"][0][i])
    print("distance:", results["distances"][0][i])
    print("metadata:", results["metadatas"][0][i])
    print("document preview:")
    print(results["documents"][0][i][:700])

rank: 1
id: 607ec1b2d3c3edda
distance: 0.4317713975906372
metadata: {'source': 'haas/Mill Spindle - Troubleshooting Guide - TG0101.html', 'chunk_index': 7, 'ext': 'html', 'type': 'troubleshooting'}
document preview:
ication system is not functioning correctly.
Inspect the spindle lubrication system.
Spindle installed incorrectly
Check Motor coupler for damage or indentations at the pin interfaces. Contact local HFO about issue including relevant pictures of the spindle to motor coupler.
The spindle temperature exceed 140° F (60° C)
The spindle lubrication system does not function correctly.
Check for a clog in the spindle lube pump and verify the type of oil.
The spindle makes an unusual noise.
The spindle drive belt is worn or damaged.
Check the spindle drive belt for damaged.
The encoder pulleys or drive belt are worn or damaged.
Check the encoder pulleys and belt.
The spindle lubrication system does 
rank: 2
id: 63c489bd31879be5
distance: 0.4553045630455017
metadata: {'type': 'troub

In [16]:
# 6. 특정 문서 source만 필터링해서 확인

target_source = "haas/Mill Chatter - Troubleshooting - TG0100.html"  # 예: haas/mill_spindle_troubleshooting.html 처럼 metadata의 source 값 입력

filtered = collection.get(
    where={"source": target_source},
    include=["documents", "metadatas"]
)

print("matched chunks:", len(filtered["ids"]))

for i in range(min(5, len(filtered["ids"]))):
    print("=" * 80)
    print("chunk id:", filtered["ids"][i])
    print("metadata:", filtered["metadatas"][i])
    print(filtered["documents"][i][:700])

matched chunks: 12
chunk id: 530c33ee610e8320
metadata: {'chunk_index': 0, 'type': 'troubleshooting', 'ext': 'html', 'source': 'haas/Mill Chatter - Troubleshooting - TG0100.html'}
Mill Chatter - Troubleshooting - TG0100
Skip To Content Button
Skip To Content
My Haas
Welcome,
Haas Tooling
MyHaas/HaasConnect
Sign In
Register
Haas Tooling
MyHaas/HaasConnect
Sign Out
Welcome,
My Machines
Latest Activity
My Quotes
My Account
My Users
Sign Out
Find Your Distributor
Select Language
English
Deutsch
Español - España
Español - México
Français
Italiano
Português
Český
Dansk
Nederlands
Magyar
Polski
Svenska
Türkçe
中文
Suomi
Norsk
الإنجليزية
български
Hrvatski
Ελληνικά
Română
Slovenský
Slovenščina
한국어
日本語
Українська
machines
Main Menu
Vertical Mills
Vertical Mills
Vertical Mills
View All
Vertical Mills
VF Series
Universal Machines
VR Series
VP-5 Prismatic
Pallet-Changing VMCs
Mini 
chunk id: 9c7d1d37437c2461
metadata: {'source': 'haas/Mill Chatter - Troubleshooting - TG0100.html', 'ext': 'html', 'ch

In [15]:
# 7. 실제 embedding vector 차원 확인

vector_sample = collection.get(
    limit=1,
    include=["embeddings", "documents", "metadatas"]
)

embedding = vector_sample["embeddings"][0]

print("embedding dimension:", len(embedding))
print("metadata:", vector_sample["metadatas"][0])
print("document preview:", vector_sample["documents"][0][:300])
print("embedding first 10 values:", embedding[:10])

embedding dimension: 1536
metadata: {'ext': 'html', 'type': 'troubleshooting', 'source': 'haas/Mill Chatter - Troubleshooting - TG0100.html', 'chunk_index': 0}
document preview: Mill Chatter - Troubleshooting - TG0100
Skip To Content Button
Skip To Content
My Haas
Welcome,
Haas Tooling
MyHaas/HaasConnect
Sign In
Register
Haas Tooling
MyHaas/HaasConnect
Sign Out
Welcome,
My Machines
Latest Activity
My Quotes
My Account
My Users
Sign Out
Find Your Distributor
Select Language

embedding first 10 values: [-0.0690918   0.02514648  0.02966309 -0.0451355  -0.01490021 -0.00626755
 -0.03683471  0.07434082  0.00541305 -0.02278137]
